# 06 - Ablation Study: Comparativa de Mecanismos del Router

Este notebook toma los embeddings (`Z_train.npy` y `Z_val.npy`) previamente extraidos usando el backbone ViT, y realiza el estudio de ablacion estipulado por la consigna comparando:
1. **ViT + Linear (Deep Learning)**: entrenado por epocas con balanceo `L_aux` (**grid** de `lr`, `alpha`).
2. **ViT + GMM**: EM / sklearn (**grid** de `covariance_type`, `reg_covar`, `n_init`).
3. **ViT + Naive Bayes**: MLE (**grid** de `var_smoothing`).
4. **ViT + k-NN**: FAISS + PCA (**grid** de `n_components` PCA y `k`).

Al final se muestra la **mejor configuración por método** en validación según **routing accuracy** y **ratio de carga** (score compuesto tipo notebook del router DL).

Nota: solo el modelo Lineal usa gradiente por epocas; el resto se ajusta en una pasada (o varias configs en bucle).


## Fase A: Importaciones, Datos y Preprocesamiento


In [5]:
!pip install -q faiss-cpu scikit-learn

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Rutas a metricas guardadas por la Fase 0 del Ablation Study en el notebook 03
FEATURE_DIR = '/content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/ablation_data_v1/'

print(f"Directorio de features esperado: {FEATURE_DIR}")


Mounted at /content/drive
Directorio de features esperado: /content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/ablation_data_v1/


In [6]:
print("Cargando los vectores CLS pre-extraidos...")

try:
    Z_train = np.load(os.path.join(FEATURE_DIR, 'Z_train.npy'))
    Z_val   = np.load(os.path.join(FEATURE_DIR, 'Z_val.npy'))
    y_train = np.load(os.path.join(FEATURE_DIR, 'y_train_expert.npy'))
    y_val   = np.load(os.path.join(FEATURE_DIR, 'y_val_expert.npy'))

    print("Arrays cargados exitosamente:")
    print(f"  Train Features : {Z_train.shape}")
    print(f"  Val Features   : {Z_val.shape}")
    print(f"  Train Labels   : {set(y_train)} -> Count: {dict(Counter(y_train))}")
    print(f"  Val Labels     : {set(y_val)} -> Count: {dict(Counter(y_val))}")
except Exception as e:
    print(f"ERROR al cargar los archivos .npy: {e}")
    print("Asegurese de haber ejecutado completamente el notebook 03 y tener datos en ablation_data_v1.")


Cargando los vectores CLS pre-extraidos...
Arrays cargados exitosamente:
  Train Features : (3556, 192)
  Val Features   : (890, 192)
  Train Labels   : {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)} -> Count: {np.int64(1): 800, np.int64(4): 445, np.int64(2): 800, np.int64(0): 800, np.int64(3): 711}
  Val Labels     : {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)} -> Count: {np.int64(3): 178, np.int64(4): 112, np.int64(1): 200, np.int64(2): 200, np.int64(0): 200}


### Utilidades: ratio de balance y selección de la mejor config

El ratio es `max(f_i)/min(f_i)` sobre predicciones duras en validación, con **piso** `1/N` si un experto queda en 0% (evita valores astronómicos).

**Criterio de selección:** maximizar `val_acc - 0.02 * max(0, val_ratio - 1.30)` (misma idea que el router DL); en empate, menor `val_ratio`.

In [7]:
import pandas as pd
from typing import Any, Dict, List, Optional, Tuple

RANDOM_STATE = 42
RATIO_TARGET = 1.30
RATIO_PENALTY = 0.02  # peso del castigo por ratio > 1.30 al elegir mejor config


def routing_ratio_from_preds_np(preds: np.ndarray, num_experts: int = 5) -> float:
    """max_i f_i / min_i f_i con piso 1/N si un experto tiene 0 predicciones."""
    preds = np.asarray(preds).astype(int).ravel()
    f = np.bincount(preds, minlength=num_experts).astype(np.float64)
    total = float(f.sum())
    if total <= 0:
        return 1.0
    frac = f / total
    floor_frac = 1.0 / total
    denom = max(float(frac.min()), floor_frac)
    return float(frac.max() / denom)


def composite_score(val_acc: float, val_ratio: float) -> float:
    return float(val_acc - RATIO_PENALTY * max(0.0, val_ratio - RATIO_TARGET))


def select_best_row(rows: List[Dict[str, Any]]) -> Tuple[Optional[Dict[str, Any]], float]:
    """Elige la fila con mayor composite_score; empate -> menor val_ratio."""
    if not rows:
        return None, -1e9
    best = max(
        rows,
        key=lambda r: (composite_score(r['val_acc'], r['val_ratio']), -r['val_ratio']),
    )
    return best, composite_score(best['val_acc'], best['val_ratio'])


print('Utilidades: routing_ratio_from_preds_np, composite_score, select_best_row OK')

Utilidades: routing_ratio_from_preds_np, composite_score, select_best_row OK


## Mecanismo A: Router Lineal (PyTorch)
Entrenamos con epocas, controlando parametricamente el Ratio de Balance gracias a L_aux.
Si el balance no logra bajar de 1.30, considere subir el parametro 'alpha'.

In [8]:
class LinearRouterHead(nn.Module):
    def __init__(self, in_features=192, num_experts=5):
        super().__init__()
        self.fc = nn.Linear(in_features, num_experts)

    def forward(self, x):
        return self.fc(x)


# --- Grid: lineal (único método con L_aux por gradiente) ---
LINEAR_GRID = {
    'lr': [1e-3, 5e-4, 2e-4],
    'alpha': [0.02, 0.05, 0.08, 0.12],
    'epochs': [120],
}

in_dim = Z_train.shape[1]
X_tr_t = torch.tensor(Z_train, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.long)
X_v_t = torch.tensor(Z_val, dtype=torch.float32)
y_v_t = torch.tensor(y_val, dtype=torch.long)

linear_rows = []
print('=== Grid Router Lineal (PyTorch) ===\n')

for lr in LINEAR_GRID['lr']:
    for alpha in LINEAR_GRID['alpha']:
        for epochs in LINEAR_GRID['epochs']:
            torch.manual_seed(RANDOM_STATE)
            router_a = LinearRouterHead(in_features=in_dim, num_experts=5).to('cpu')
            optimizer = torch.optim.Adam(router_a.parameters(), lr=lr)

            best_snap = {'val_acc': 0.0, 'val_ratio': float('inf'), 'epoch': 0, 'score': -1e9}

            for epoch in range(1, epochs + 1):
                router_a.train()
                optimizer.zero_grad()
                logits = router_a(X_tr_t)
                loss_task = F.cross_entropy(logits, y_tr_t)
                probs = F.softmax(logits, dim=1)
                mean_probs = probs.mean(dim=0)
                hard_preds = logits.argmax(dim=1)
                counts = torch.bincount(hard_preds, minlength=5)
                f_i = counts.float() / len(hard_preds)
                l_aux = alpha * 5.0 * torch.sum(mean_probs * f_i)
                loss_total = loss_task + l_aux
                loss_total.backward()
                optimizer.step()

                if epoch % 20 == 0 or epoch == epochs:
                    router_a.eval()
                    with torch.no_grad():
                        val_logits = router_a(X_v_t)
                        val_preds = val_logits.argmax(dim=1).cpu().numpy()
                        val_acc = float(np.mean(val_preds == y_v_t.numpy()))
                        val_ratio = routing_ratio_from_preds_np(val_preds, 5)
                        sc = composite_score(val_acc, val_ratio)
                        if sc > best_snap['score'] or (
                            abs(sc - best_snap['score']) < 1e-9 and val_ratio < best_snap['val_ratio']
                        ):
                            best_snap = {
                                'val_acc': val_acc,
                                'val_ratio': val_ratio,
                                'epoch': epoch,
                                'score': sc,
                            }

            linear_rows.append(
                {
                    'method': 'Linear',
                    'lr': lr,
                    'alpha': alpha,
                    'epochs': epochs,
                    'best_epoch': best_snap['epoch'],
                    'val_acc': best_snap['val_acc'],
                    'val_ratio': best_snap['val_ratio'],
                    'composite': best_snap['score'],
                }
            )
            print(
                f"lr={lr:g} alpha={alpha:g} ep={epochs} | "
                f"mejor@ep{best_snap['epoch']} val_acc={best_snap['val_acc']:.4f} "
                f"val_ratio={best_snap['val_ratio']:.3f} score={best_snap['score']:.4f}"
            )

df_linear = pd.DataFrame(linear_rows)
best_linear, _ = select_best_row(linear_rows)
print('\n--- Mejor config Lineal ---')
print(best_linear)


=== Grid Router Lineal (PyTorch) ===

lr=0.001 alpha=0.02 ep=120 | mejor@ep120 val_acc=0.8326 val_ratio=2.241 score=0.8138
lr=0.001 alpha=0.05 ep=120 | mejor@ep120 val_acc=0.8270 val_ratio=2.179 score=0.8094
lr=0.001 alpha=0.08 ep=120 | mejor@ep120 val_acc=0.8236 val_ratio=2.134 score=0.8069
lr=0.001 alpha=0.12 ep=120 | mejor@ep120 val_acc=0.8213 val_ratio=2.045 score=0.8065
lr=0.0005 alpha=0.02 ep=120 | mejor@ep120 val_acc=0.8124 val_ratio=2.045 score=0.7975
lr=0.0005 alpha=0.05 ep=120 | mejor@ep120 val_acc=0.8090 val_ratio=1.946 score=0.7961
lr=0.0005 alpha=0.08 ep=120 | mejor@ep120 val_acc=0.8034 val_ratio=1.857 score=0.7922
lr=0.0005 alpha=0.12 ep=120 | mejor@ep120 val_acc=0.7989 val_ratio=1.839 score=0.7881
lr=0.0002 alpha=0.02 ep=120 | mejor@ep120 val_acc=0.7820 val_ratio=2.396 score=0.7601
lr=0.0002 alpha=0.05 ep=120 | mejor@ep120 val_acc=0.7798 val_ratio=2.194 score=0.7619
lr=0.0002 alpha=0.08 ep=120 | mejor@ep120 val_acc=0.7764 val_ratio=2.019 score=0.7620
lr=0.0002 alpha=0.12

## Mecanismo B: Gaussian Mixture Model (GMM)
Metodo parametrico estadistico. Las semillas medias (means_init) se inicializan con la media empirica de entrenamiento. Usamos `covariance_type='diag'` para asegurar la estabilidad matematica segun sugiere la consigna.

In [9]:
from sklearn.mixture import GaussianMixture

print('=== Grid GMM (sklearn) ===\n')
print('Nota: mismas componentes que expertos; means_init = centro por experto en train.\n')

expert_centers = np.array([Z_train[y_train == i].mean(axis=0) for i in range(5)])

# Grid: diag (estable en alta dim); full solo con reg alto para no explotar en CPU
GMM_GRID = []
for reg in (1e-7, 1e-5, 1e-3):
    for n_init in (3, 5, 10):
        GMM_GRID.append(dict(covariance_type='diag', reg_covar=reg, n_init=n_init, max_iter=300))
for reg in (1e-4, 1e-3):
    GMM_GRID.append(dict(covariance_type='full', reg_covar=reg, n_init=5, max_iter=200))

gmm_rows = []
for cfg in GMM_GRID:
    try:
        gmm = GaussianMixture(
            n_components=5,
            covariance_type=cfg['covariance_type'],
            means_init=expert_centers.copy(),
            reg_covar=cfg['reg_covar'],
            n_init=cfg['n_init'],
            max_iter=cfg['max_iter'],
            random_state=RANDOM_STATE,
        )
        gmm.fit(Z_train)
        gmm_preds = gmm.predict_proba(Z_val).argmax(axis=1)
        val_acc = float(np.mean(gmm_preds == y_val))
        val_ratio = routing_ratio_from_preds_np(gmm_preds, 5)
        sc = composite_score(val_acc, val_ratio)
        gmm_rows.append(
            {
                'method': 'GMM',
                **cfg,
                'val_acc': val_acc,
                'val_ratio': val_ratio,
                'composite': sc,
            }
        )
        print(
            f"cov={cfg['covariance_type']} reg={cfg['reg_covar']:.0e} n_init={cfg['n_init']} | "
            f"val_acc={val_acc:.4f} val_ratio={val_ratio:.3f} score={sc:.4f}"
        )
    except Exception as e:
        print(f"SKIP {cfg}: {e}")

df_gmm = pd.DataFrame(gmm_rows)
best_gmm, _ = select_best_row(gmm_rows)
print('\n--- Mejor config GMM ---')
print(best_gmm)


=== Grid GMM (sklearn) ===

Nota: mismas componentes que expertos; means_init = centro por experto en train.

cov=diag reg=1e-07 n_init=3 | val_acc=0.6056 val_ratio=345.000 score=-6.2684
cov=diag reg=1e-07 n_init=5 | val_acc=0.6056 val_ratio=345.000 score=-6.2684
cov=diag reg=1e-07 n_init=10 | val_acc=0.6056 val_ratio=345.000 score=-6.2684
cov=diag reg=1e-05 n_init=3 | val_acc=0.6056 val_ratio=345.000 score=-6.2684
cov=diag reg=1e-05 n_init=5 | val_acc=0.6056 val_ratio=345.000 score=-6.2684
cov=diag reg=1e-05 n_init=10 | val_acc=0.6056 val_ratio=345.000 score=-6.2684
cov=diag reg=1e-03 n_init=3 | val_acc=0.8011 val_ratio=2.027 score=0.7866
cov=diag reg=1e-03 n_init=5 | val_acc=0.8011 val_ratio=2.027 score=0.7866
cov=diag reg=1e-03 n_init=10 | val_acc=0.8011 val_ratio=2.027 score=0.7866
cov=full reg=1e-04 n_init=5 | val_acc=0.6539 val_ratio=479.000 score=-8.9001
cov=full reg=1e-03 n_init=5 | val_acc=0.5382 val_ratio=438.000 score=-8.1958

--- Mejor config GMM ---
{'method': 'GMM', 'cova

## Mecanismo C: Gaussian Naive Bayes
Desempeno de linea base (modelo analitico en una pasada) asumiendo independencia entre dimensiones.

In [10]:
from sklearn.naive_bayes import GaussianNB

print('=== Grid GaussianNB ===\n')

NB_VAR_SMOOTHING = [1e-11, 1e-9, 1e-7, 1e-5, 1e-3]

nb_rows = []
for vs in NB_VAR_SMOOTHING:
    nb = GaussianNB(var_smoothing=vs)
    nb.fit(Z_train, y_train)
    nb_preds = nb.predict_proba(Z_val).argmax(axis=1)
    val_acc = float(np.mean(nb_preds == y_val))
    val_ratio = routing_ratio_from_preds_np(nb_preds, 5)
    sc = composite_score(val_acc, val_ratio)
    nb_rows.append(
        {
            'method': 'GaussianNB',
            'var_smoothing': vs,
            'val_acc': val_acc,
            'val_ratio': val_ratio,
            'composite': sc,
        }
    )
    print(f"var_smoothing={vs:.0e} | val_acc={val_acc:.4f} val_ratio={val_ratio:.3f} score={sc:.4f}")

df_nb = pd.DataFrame(nb_rows)
best_nb, _ = select_best_row(nb_rows)
print('\n--- Mejor config GaussianNB ---')
print(best_nb)


=== Grid GaussianNB ===

var_smoothing=1e-11 | val_acc=0.8157 val_ratio=2.402 score=0.7937
var_smoothing=1e-09 | val_acc=0.8157 val_ratio=2.402 score=0.7937
var_smoothing=1e-07 | val_acc=0.8157 val_ratio=2.402 score=0.7937
var_smoothing=1e-05 | val_acc=0.8157 val_ratio=2.402 score=0.7937
var_smoothing=1e-03 | val_acc=0.8202 val_ratio=2.438 score=0.7975

--- Mejor config GaussianNB ---
{'method': 'GaussianNB', 'var_smoothing': 0.001, 'val_acc': 0.8202247191011236, 'val_ratio': 2.4375, 'composite': 0.7974747191011236}


## Mecanismo D: k-NN (K-Nearest Neighbors con FAISS)
Usando FAISS y reduciendo la dimensionalidad a 32 previamente con PCA para mitigar la Maldicion de Dimensionalidad sugerida en la orientacion del curso.

In [11]:
import faiss
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

print('=== Grid k-NN (FAISS + PCA + coseno) ===\n')

n_samples, n_feat = Z_train.shape
max_pca = min(96, n_feat - 1, max(2, n_samples - 1))
PCA_DIMS = [d for d in (16, 32, 64, max_pca) if 2 <= d <= max_pca]
PCA_DIMS = sorted(set(int(x) for x in PCA_DIMS))
K_LIST = [3, 5, 7, 11, 15]
k_max = min(max(K_LIST), max(1, n_samples - 1))

knn_rows = []
for n_comp in PCA_DIMS:
    if n_comp >= Z_train.shape[1]:
        continue
    pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
    Z_tr_pca = pca.fit_transform(Z_train)
    Z_v_pca = pca.transform(Z_val)
    Z_tr_norm = normalize(Z_tr_pca, norm='l2').astype('float32')
    Z_v_norm = normalize(Z_v_pca, norm='l2').astype('float32')
    dim_pca = Z_tr_norm.shape[1]
    index = faiss.IndexFlatIP(dim_pca)
    index.add(Z_tr_norm)

    for k in K_LIST:
        k_eff = int(min(k, k_max))
        if k_eff < 1:
            continue
        dist, I = index.search(Z_v_norm, k_eff)
        neighbor_labels = y_train[I]
        knn_preds = np.apply_along_axis(
            lambda x: np.bincount(x, minlength=5).argmax(),
            axis=1,
            arr=neighbor_labels,
        )
        val_acc = float(np.mean(knn_preds == y_val))
        val_ratio = routing_ratio_from_preds_np(knn_preds, 5)
        sc = composite_score(val_acc, val_ratio)
        knn_rows.append(
            {
                'method': 'kNN_FAISS',
                'pca_dim': n_comp,
                'k': k_eff,
                'val_acc': val_acc,
                'val_ratio': val_ratio,
                'composite': sc,
            }
        )
        print(
            f"PCA={n_comp} k={k_eff} | val_acc={val_acc:.4f} val_ratio={val_ratio:.3f} score={sc:.4f}"
        )

df_knn = pd.DataFrame(knn_rows)
best_knn, _ = select_best_row(knn_rows)
print('\n--- Mejor config k-NN ---')
print(best_knn)


=== Grid k-NN (FAISS + PCA + coseno) ===

PCA=16 k=3 | val_acc=0.9371 val_ratio=1.911 score=0.9249
PCA=16 k=5 | val_acc=0.9404 val_ratio=1.884 score=0.9288
PCA=16 k=7 | val_acc=0.9472 val_ratio=1.866 score=0.9359
PCA=16 k=11 | val_acc=0.9449 val_ratio=1.848 score=0.9340
PCA=16 k=15 | val_acc=0.9404 val_ratio=1.848 score=0.9295
PCA=32 k=3 | val_acc=0.9596 val_ratio=1.875 score=0.9481
PCA=32 k=5 | val_acc=0.9596 val_ratio=1.866 score=0.9482
PCA=32 k=7 | val_acc=0.9528 val_ratio=1.866 score=0.9415
PCA=32 k=11 | val_acc=0.9494 val_ratio=1.866 score=0.9381
PCA=32 k=15 | val_acc=0.9506 val_ratio=1.857 score=0.9394
PCA=64 k=3 | val_acc=0.9596 val_ratio=1.848 score=0.9486
PCA=64 k=5 | val_acc=0.9551 val_ratio=1.893 score=0.9432
PCA=64 k=7 | val_acc=0.9517 val_ratio=1.902 score=0.9396
PCA=64 k=11 | val_acc=0.9494 val_ratio=1.884 score=0.9378
PCA=64 k=15 | val_acc=0.9483 val_ratio=1.875 score=0.9368
PCA=96 k=3 | val_acc=0.9629 val_ratio=1.839 score=0.9521
PCA=96 k=5 | val_acc=0.9528 val_ratio=1.

## Resumen: mejor configuración por método

Tabla compacta: **routing accuracy** y **ratio** en validación, y **score** = `acc - 0.02 * max(0, ratio - 1.30)`.

Los `DataFrame` completos (`df_linear`, `df_gmm`, `df_nb`, `df_knn`) guardan **todas** las configs evaluadas.

In [12]:
summary_rows = []
for name, row in [
    ('Linear', best_linear),
    ('GMM', best_gmm),
    ('GaussianNB', best_nb),
    ('kNN_FAISS', best_knn),
]:
    if row is None:
        continue
    summary_rows.append(
        {
            'method': name,
            'val_acc': row['val_acc'],
            'val_ratio': row['val_ratio'],
            'composite': composite_score(row['val_acc'], row['val_ratio']),
            'best_config': {k: v for k, v in row.items() if k not in ('val_acc', 'val_ratio', 'composite', 'method')},
        }
    )

df_summary = pd.DataFrame(summary_rows)
if len(df_summary):
    df_summary = df_summary.sort_values('composite', ascending=False).reset_index(drop=True)

print(df_summary.to_string(index=False))

_candidates = [r for r in [best_linear, best_gmm, best_nb, best_knn] if r is not None]
best_overall, best_overall_score = select_best_row(_candidates)
print('\n=== Mejor método global (según composite en val) ===')
print(f'score={best_overall_score:.4f}')
print(best_overall)

    method  val_acc  val_ratio  composite                                                                   best_config
 kNN_FAISS 0.962921   1.839286   0.952136                                                       {'pca_dim': 96, 'k': 3}
    Linear 0.832584   2.241071   0.813763                {'lr': 0.001, 'alpha': 0.02, 'epochs': 120, 'best_epoch': 120}
GaussianNB 0.820225   2.437500   0.797475                                                      {'var_smoothing': 0.001}
       GMM 0.801124   2.026786   0.786588 {'covariance_type': 'diag', 'reg_covar': 0.001, 'n_init': 3, 'max_iter': 300}

=== Mejor método global (según composite en val) ===
score=0.9521
{'method': 'kNN_FAISS', 'pca_dim': 96, 'k': 3, 'val_acc': 0.9629213483146067, 'val_ratio': 1.8392857142857142, 'composite': 0.9521356340288925}
